# YOLO11n-seg用データセット準備

このノートブックは、既存の6クラスマスクからYOLO11n-seg用のアノテーションを生成します。

## 処理内容

1. **輪郭抽出**: 各クラスのマスクから輪郭を抽出
2. **100点サンプリング**: 輪郭に沿って100点を均等に配置
3. **YOLO形式変換**: YOLOセグメンテーション形式に変換

## YOLO形式

- **画像**: `Images/images/{filename}`
- **アノテーション**: `Images/labels/{stem}.txt`
  - 各行: `class_id x1 y1 x2 y2 ... x100 y100`（正規化座標 0-1）
  - 各クラス（0-5）ごとに1行

## 6クラス定義

```
0: background   = lid外 ∩ iris外 ∩ pupil外（スキップ）
1: conj         = lid内 ∩ iris外 ∩ pupil外
2: iris_vis     = lid内 ∩ iris内 ∩ pupil外
3: iris_occ     = lid外 ∩ iris内 ∩ pupil外
4: pupil_vis    = lid内 ∩ iris内 ∩ pupil内
5: pupil_occ    = lid外 ∩ iris内 ∩ pupil内
```

**注意**: background（クラス0）はスキップします（YOLOでは背景クラスは通常アノテーション不要）


## 1. インポート・設定


In [25]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import json

# ===== パス設定 =====
IMAGES_DIR = Path("Images/images")
LABEL_SEG_DIR = Path("Images/labels_seg")
LABEL_OBB_DIR = Path("Images/labels_obb")

# YOLOラベルファイルの出力先
# 【重要】YOLOは画像パスから自動的にラベルパスを推測します
# 画像が Images/images/xxx.jpg の場合、ラベルは Images/labels/xxx.txt を探します
# そのため、ラベルファイルは必ず Images/labels/ に保存する必要があります
YOLO_LABELS_DIR = Path("Images/labels")
YOLO_LABELS_DIR.mkdir(parents=True, exist_ok=True)

# YOLO設定ファイル（yaml/txt）の出力先
# データセット設定ファイル（dataset_yolo11.yaml）やFold分割リスト（yolo11_train_fold*.txt等）
# は Ablation_YOLO フォルダにまとめて保存します
ABLATION_YOLO_DIR = Path("Ablation_YOLO")
ABLATION_YOLO_DIR.mkdir(parents=True, exist_ok=True)

# パラメータ
IMAGE_SIZE = 512  # 画像サイズ
NUM_POINTS = 100  # 輪郭に沿って配置する点の数

print(f"✓ セットアップ完了")
print(f"  - ラベル出力先: {YOLO_LABELS_DIR}")
print(f"  - YOLO設定ファイル出力先: {ABLATION_YOLO_DIR}")
print(f"  - 画像サイズ: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"  - 点の数: {NUM_POINTS}")


✓ セットアップ完了
  - ラベル出力先: Images\labels
  - YOLO設定ファイル出力先: Ablation_YOLO
  - 画像サイズ: 512×512
  - 点の数: 100


## 2. 輪郭抽出・100点サンプリング関数


In [26]:
def sample_points_along_contour(contour, num_points=100):
    """
    輪郭に沿って均等に点をサンプリング
    
    Args:
        contour: OpenCV輪郭（形状: (N, 1, 2)）
        num_points: サンプリングする点の数
    
    Returns:
        np.ndarray: (num_points, 2) の点座標（x, y）
    """
    if len(contour) < 2:
        return None
    
    # 輪郭を1次元配列に変換
    contour_1d = contour.reshape(-1, 2).astype(np.float32)
    
    # 輪郭の周囲長を計算（累積距離）
    distances = np.zeros(len(contour_1d))
    for i in range(1, len(contour_1d)):
        dist = np.linalg.norm(contour_1d[i] - contour_1d[i-1])
        distances[i] = distances[i-1] + dist
    
    # 閉じた輪郭の場合、最後の点から最初の点への距離も考慮
    total_length = distances[-1]
    if total_length > 0:
        # 閉じた輪郭として扱う（最後→最初への接続を考慮）
        last_to_first = np.linalg.norm(contour_1d[0] - contour_1d[-1])
        if last_to_first < 10:  # 近接していれば閉じた輪郭として扱う
            total_length += last_to_first
    
    if total_length == 0:
        return None
    
    # 均等に点をサンプリング
    sampled_points = []
    for i in range(num_points):
        target_dist = (total_length * i) / num_points
        
        # 距離配列から該当するインデックスを探す
        idx = np.searchsorted(distances, target_dist, side='right')
        idx = min(idx, len(contour_1d) - 1)
        
        if idx == 0:
            point = contour_1d[0]
        elif idx >= len(distances):
            point = contour_1d[-1]
        else:
            # 線形補間
            dist_before = distances[idx - 1]
            dist_after = distances[idx]
            if dist_after > dist_before:
                alpha = (target_dist - dist_before) / (dist_after - dist_before)
                point = contour_1d[idx - 1] * (1 - alpha) + contour_1d[idx] * alpha
            else:
                point = contour_1d[idx]
        
        sampled_points.append(point)
    
    return np.array(sampled_points, dtype=np.float32)


def extract_contours_from_mask(mask, min_area=10):
    """
    マスクから輪郭を抽出
    
    Args:
        mask: バイナリマスク（uint8, 0 or 255）
        min_area: 最小面積（これより小さい輪郭は無視）
    
    Returns:
        list: 輪郭のリスト（各輪郭は (N, 1, 2) 形状）
    """
    # 2値化
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    
    # 輪郭抽出（RETR_EXTERNAL: 外側の輪郭のみ）
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    
    # 面積でフィルタリング
    filtered_contours = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_area:
            filtered_contours.append(cnt)
    
    return filtered_contours

print("✓ 輪郭抽出・サンプリング関数を定義しました")


✓ 輪郭抽出・サンプリング関数を定義しました


## 3. 6クラスマスクからYOLO形式への変換


In [27]:
# sixcls.pngのBGR色からクラスIDへのマッピング
SIXCLS_BGR_TO_ID = {
    (0, 0, 0): 0,         # background - 黒（スキップ）
    (255, 0, 0): 1,       # conj (lid) - 青(BGR)
    (0, 255, 0): 2,       # iris_vis - 緑
    (0, 0, 255): 3,       # iris_occ - 赤(BGR)
    (0, 255, 255): 4,     # pupil_vis - 黄(BGR)
    (255, 0, 255): 5,     # pupil_occ - マゼンタ(BGR)
}

def convert_sixcls_to_mask(sixcls_img, class_id):
    """
    sixcls.pngから特定クラスのマスクを抽出
    
    Args:
        sixcls_img: sixcls.png画像（BGR形式）
        class_id: 抽出するクラスID（0-5）
    
    Returns:
        np.ndarray: バイナリマスク（uint8, 0 or 255）
    """
    # クラスIDに対応する色を取得
    target_color = None
    for bgr_color, cid in SIXCLS_BGR_TO_ID.items():
        if cid == class_id:
            target_color = bgr_color
            break
    
    if target_color is None:
        return np.zeros(sixcls_img.shape[:2], dtype=np.uint8)
    
    # 該当色のピクセルを抽出
    mask = np.all(sixcls_img == target_color, axis=2).astype(np.uint8) * 255
    return mask


def mask_to_yolo_format(mask, class_id, image_size=512, num_points=100):
    """
    マスクをYOLO形式（正規化座標）に変換
    
    Args:
        mask: バイナリマスク（uint8, 0 or 255）
        class_id: クラスID
        image_size: 画像サイズ
        num_points: 輪郭に沿って配置する点の数
    
    Returns:
        list: YOLO形式の行リスト（各行: "class_id x1 y1 x2 y2 ..."）
    """
    yolo_lines = []
    
    # 輪郭抽出
    contours = extract_contours_from_mask(mask, min_area=10)
    
    # 各輪郭を処理
    for contour in contours:
        # 100点サンプリング
        points = sample_points_along_contour(contour, num_points)
        
        if points is None or len(points) == 0:
            continue
        
        # 正規化座標に変換（0-1範囲）
        normalized_points = points / image_size
        
        # YOLO形式の行を作成
        # 形式: "class_id x1 y1 x2 y2 ... x100 y100"
        yolo_line = f"{class_id}"
        for x, y in normalized_points:
            yolo_line += f" {x:.6f} {y:.6f}"
        
        yolo_lines.append(yolo_line)
    
    return yolo_lines

print("✓ 変換関数を定義しました")


✓ 変換関数を定義しました


In [28]:
# 画像リストを読み込み
df = pd.read_csv('image_metadata.csv')
print(f"総画像数: {len(df)}")

# 統計情報
stats = {
    'total': 0,
    'success': 0,
    'failed': 0,
    'class_counts': {i: 0 for i in range(1, 6)}  # クラス0（background）はスキップ
}

print("\n処理を開始します...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    filename = row['filename']
    stem = Path(filename).stem
    
    stats['total'] += 1
    
    # sixcls.pngを読み込み
    sixcls_path = LABEL_SEG_DIR / f"{stem}_sixcls.png"
    if not sixcls_path.exists():
        stats['failed'] += 1
        continue
    
    sixcls_img = cv2.imread(str(sixcls_path))
    if sixcls_img is None:
        stats['failed'] += 1
        continue
    
    # 512×512にリサイズ（念のため）
    if sixcls_img.shape[:2] != (IMAGE_SIZE, IMAGE_SIZE):
        sixcls_img = cv2.resize(sixcls_img, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
    
    # YOLO形式のアノテーションファイルを作成
    yolo_lines = []
    
    # 各クラス（1-5）を処理（クラス0は背景なのでスキップ）
    for class_id in range(1, 6):
        # クラスマスクを抽出
        class_mask = convert_sixcls_to_mask(sixcls_img, class_id)
        
        # YOLO形式に変換
        class_yolo_lines = mask_to_yolo_format(class_mask, class_id, IMAGE_SIZE, NUM_POINTS)
        
        if len(class_yolo_lines) > 0:
            yolo_lines.extend(class_yolo_lines)
            stats['class_counts'][class_id] += len(class_yolo_lines)
    
    # アノテーションファイルを保存（Images/labels/ に保存）
    # 【重要】YOLOは画像パスから自動的にラベルパスを推測するため、
    # 画像が Images/images/xxx.jpg の場合、ラベルは Images/labels/xxx.txt を探します
    yolo_label_path = YOLO_LABELS_DIR / f"{stem}.txt"
    if len(yolo_lines) > 0:
        with open(yolo_label_path, 'w') as f:
            f.write('\n'.join(yolo_lines) + '\n')
        stats['success'] += 1
    else:
        # アノテーションが空の場合は空ファイルを作成
        yolo_label_path.touch()
        stats['success'] += 1

print("\n" + "=" * 80)
print("✅ 処理完了")
print("=" * 80)
print(f"総画像数: {stats['total']}")
print(f"成功: {stats['success']}")
print(f"失敗: {stats['failed']}")
print(f"\nクラス別アノテーション数:")
for class_id, count in stats['class_counts'].items():
    class_names = {1: 'conj', 2: 'iris_vis', 3: 'iris_occ', 4: 'pupil_vis', 5: 'pupil_occ'}
    print(f"  クラス{class_id} ({class_names[class_id]}): {count}個")


総画像数: 1992

処理を開始します...


Processing: 100%|██████████| 1992/1992 [01:28<00:00, 22.54it/s]


✅ 処理完了
総画像数: 1992
成功: 1992
失敗: 0

クラス別アノテーション数:
  クラス1 (conj): 3104個
  クラス2 (iris_vis): 1992個
  クラス3 (iris_occ): 3001個
  クラス4 (pupil_vis): 1975個
  クラス5 (pupil_occ): 329個


## 5. データセット構造の確認


In [29]:
# サンプルアノテーションファイルを確認
sample_files = list(YOLO_LABELS_DIR.glob("*.txt"))[:5]
print(f"サンプルファイル数: {len(sample_files)}")

for label_file in sample_files:
    print(f"\n--- {label_file.name} ---")
    with open(label_file, 'r') as f:
        lines = f.readlines()
        print(f"行数: {len(lines)}")
        if len(lines) > 0:
            # 最初の行を表示（最初の20点のみ）
            first_line = lines[0].strip()
            parts = first_line.split()
            class_id = parts[0]
            points_count = (len(parts) - 1) // 2
            print(f"  クラスID: {class_id}, 点の数: {points_count}")
            if points_count > 0:
                # 最初の10点を表示
                print(f"  最初の10点: {parts[1:21]}")


サンプルファイル数: 5

--- 1-20141126-38-091804_eb568e2ac952f8be45ec0ac9ae800120b7c988b60ac499ca87306986d218f554_L.txt ---
行数: 7
  クラスID: 1, 点の数: 100
  最初の10点: ['0.630859', '0.472656', '0.629723', '0.477699', '0.627206', '0.482169', '0.626070', '0.487211', '0.623553', '0.491682', '0.621094', '0.496176', '0.618518', '0.500623', '0.616001', '0.505093', '0.613281', '0.509479', '0.609585', '0.513462']

--- 1-20141126-38-091804_eb568e2ac952f8be45ec0ac9ae800120b7c988b60ac499ca87306986d218f554_R.txt ---
行数: 6
  クラスID: 1, 点の数: 100
  最初の10点: ['0.332031', '0.486328', '0.328169', '0.488281', '0.323498', '0.488281', '0.318827', '0.488281', '0.314156', '0.488281', '0.309485', '0.488281', '0.304814', '0.488281', '0.300952', '0.490234', '0.296281', '0.490234', '0.292419', '0.492188']

--- 1-20150121-38-142903_6e60b2355e174936406b708cf171e424300de779d4b8ae8e3aebb1a9de9905e6_L.txt ---
行数: 6
  クラスID: 1, 点の数: 100
  最初の10点: ['0.613281', '0.414062', '0.616844', '0.419578', '0.619141', '0.425618', '0.623047', '0.430

## 6. YOLO形式データセット用の設定ファイル作成


In [30]:
# ===== YOLO用のデータセット設定ファイル（yaml形式）を作成 =====
# このファイルは学習時に使用します
# 【保存先】Ablation_YOLO/dataset_yolo11.yaml

dataset_yaml = """
# 6クラス眼部セグメンテーションデータセット
path: .  # プロジェクトルート
train: Images/images  # 画像ディレクトリ（fold分割は別途管理）
val: Images/images    # 画像ディレクトリ（fold分割は別途管理）

# クラス名
names:
  0: background
  1: conj
  2: iris_vis
  3: iris_occ
  4: pupil_vis
  5: pupil_occ

# クラス数（backgroundを含む）
nc: 6
"""

# Ablation_YOLOフォルダに保存（YOLO関連ファイルをまとめて管理）
yaml_file = ABLATION_YOLO_DIR / 'dataset_yolo11.yaml'
with open(yaml_file, 'w', encoding='utf-8') as f:
    f.write(dataset_yaml.strip())

print(f"✅ {yaml_file} を作成しました")
print("   このファイルは学習時に使用します")


✅ Ablation_YOLO\dataset_yolo11.yaml を作成しました
   このファイルは学習時に使用します


In [ ]:
# Method3（U-Net）vs SegFormer 比較（セル単独実行でも動くように self-contained）
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Method3（U-Net）側：最新の summary CSV ----
method3_files = sorted(results_dir.glob("cv_eval_summary_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not method3_files:
    print("⚠️ Method3の結果ファイル（results/cv_eval_summary_*.csv）が見つかりません。")
else:
    method3_results = pd.read_csv(method3_files[0])
    if 'Method' not in method3_results.columns or 'Method3' not in set(method3_results['Method'].astype(str)):
        print(f"⚠️ Method3行が見つかりません: {method3_files[0]}")
    else:
        method3_row = method3_results[method3_results['Method'].astype(str) == 'Method3'].iloc[0]

        # ---- SegFormer側：どの手法の結果と比べるか ----
        # raw / outerarc / fullmax / ransac_whole / ransac_arc
        SEGFORMER_COMPARE_MODE = 'fullmax'

        # まず per-image CSV（推奨）から平均±SD（画像単位）を作る。無ければfold平均CSVにfallback。
        per_path = sorted(results_dir.glob(f"segformer_eval_perimage_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
        fold_path = sorted(results_dir.glob(f"segformer_eval_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

        seg_source = None
        if per_path:
            df_seg = pd.read_csv(per_path[0])
            seg_source = per_path[0]
        elif fold_path:
            df_seg = pd.read_csv(fold_path[0])
            seg_source = fold_path[0]
        else:
            df_seg = None

        if df_seg is None:
            print(f"⚠️ SegFormerの結果CSVが見つかりません: mode={SEGFORMER_COMPARE_MODE}")
        else:
            # df_seg は列: eyelid/iris/pupil/mean を想定
            for col in ['eyelid','iris','pupil','mean']:
                if col not in df_seg.columns:
                    raise ValueError(f"SegFormer CSVに列 '{col}' がありません: {seg_source}")

            summary = {
                'Eyelid_mean': float(df_seg['eyelid'].mean()),
                'Eyelid_std': float(df_seg['eyelid'].std(ddof=1)),
                'Iris_mean': float(df_seg['iris'].mean()),
                'Iris_std': float(df_seg['iris'].std(ddof=1)),
                'Pupil_mean': float(df_seg['pupil'].mean()),
                'Pupil_std': float(df_seg['pupil'].std(ddof=1)),
                'Total_mean': float(df_seg['mean'].mean()),
                'Total_std': float(df_seg['mean'].std(ddof=1)),
            }

            print("\nMethod3 vs SegFormer 比較")
            print("=" * 80)
            print(f"SegFormer mode: {SEGFORMER_COMPARE_MODE}")
            print(f"SegFormer CSV : {seg_source.resolve()}")
            print("-" * 80)
            print(f"{'Metric':<15} {'Method3 (U-Net)':<22} {'SegFormer':<22} {'Diff(S-M3)':<12}")
            print("-" * 80)

            metrics = [
                ('Eyelid', 'Eyelid_mean', 'Eyelid_std'),
                ('Iris', 'Iris_mean', 'Iris_std'),
                ('Pupil', 'Pupil_mean', 'Pupil_std'),
                ('Mean', 'Total_mean', 'Total_std'),
            ]

            for name, mean_key, std_key in metrics:
                m3_mean = float(method3_row[mean_key])
                m3_std = float(method3_row[std_key])
                seg_mean = float(summary[mean_key])
                seg_std = float(summary[std_key])
                diff = seg_mean - m3_mean
                print(f"{name:<15} {m3_mean:.4f} ± {m3_std:.4f}      {seg_mean:.4f} ± {seg_std:.4f}      {diff:+.4f}")

            print("=" * 80)

            comparison = {
                'Method': ['Method3 (U-Net)', f"SegFormer ({SEGFORMER_COMPARE_MODE})"],
                'Eyelid_mean': [float(method3_row['Eyelid_mean']), summary['Eyelid_mean']],
                'Eyelid_std': [float(method3_row['Eyelid_std']), summary['Eyelid_std']],
                'Iris_mean': [float(method3_row['Iris_mean']), summary['Iris_mean']],
                'Iris_std': [float(method3_row['Iris_std']), summary['Iris_std']],
                'Pupil_mean': [float(method3_row['Pupil_mean']), summary['Pupil_mean']],
                'Pupil_std': [float(method3_row['Pupil_std']), summary['Pupil_std']],
                'Total_mean': [float(method3_row['Total_mean']), summary['Total_mean']],
                'Total_std': [float(method3_row['Total_std']), summary['Total_std']],
            }
            comparison_df = pd.DataFrame(comparison)
            out_path = results_dir / f"ablation_comparison_{timestamp}.csv"
            comparison_df.to_csv(out_path, index=False)
            print(f"\n✅ 比較結果を保存しました: {out_path.resolve()}")



📊 Method3 vs SegFormer 比較
Metric          Method3 (U-Net)      SegFormer            差分             
--------------------------------------------------------------------------------


NameError: name 'summary' is not defined

In [ ]:
# Method3（U-Net）vs SegFormer 比較（セル単独実行でも動くように self-contained）
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Method3（U-Net）側：最新の summary CSV ----
method3_files = sorted(results_dir.glob("cv_eval_summary_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not method3_files:
    print("⚠️ Method3の結果ファイル（results/cv_eval_summary_*.csv）が見つかりません。")
else:
    method3_results = pd.read_csv(method3_files[0])
    if 'Method' not in method3_results.columns or 'Method3' not in set(method3_results['Method'].astype(str)):
        print(f"⚠️ Method3行が見つかりません: {method3_files[0]}")
    else:
        method3_row = method3_results[method3_results['Method'].astype(str) == 'Method3'].iloc[0]

        # ---- SegFormer側：どの手法の結果と比べるか ----
        # raw / outerarc / fullmax / ransac_whole / ransac_arc
        SEGFORMER_COMPARE_MODE = 'fullmax'

        # まず per-image CSV（推奨）から平均±SD（画像単位）を作る。無ければfold平均CSVにfallback。
        per_path = sorted(results_dir.glob(f"segformer_eval_perimage_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
        fold_path = sorted(results_dir.glob(f"segformer_eval_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

        seg_source = None
        if per_path:
            df_seg = pd.read_csv(per_path[0])
            seg_source = per_path[0]
        elif fold_path:
            df_seg = pd.read_csv(fold_path[0])
            seg_source = fold_path[0]
        else:
            df_seg = None

        if df_seg is None:
            print(f"⚠️ SegFormerの結果CSVが見つかりません: mode={SEGFORMER_COMPARE_MODE}")
        else:
            # df_seg は列: eyelid/iris/pupil/mean を想定
            for col in ['eyelid','iris','pupil','mean']:
                if col not in df_seg.columns:
                    raise ValueError(f"SegFormer CSVに列 '{col}' がありません: {seg_source}")

            summary = {
                'Eyelid_mean': float(df_seg['eyelid'].mean()),
                'Eyelid_std': float(df_seg['eyelid'].std(ddof=1)),
                'Iris_mean': float(df_seg['iris'].mean()),
                'Iris_std': float(df_seg['iris'].std(ddof=1)),
                'Pupil_mean': float(df_seg['pupil'].mean()),
                'Pupil_std': float(df_seg['pupil'].std(ddof=1)),
                'Total_mean': float(df_seg['mean'].mean()),
                'Total_std': float(df_seg['mean'].std(ddof=1)),
            }

            print("\nMethod3 vs SegFormer 比較")
            print("=" * 80)
            print(f"SegFormer mode: {SEGFORMER_COMPARE_MODE}")
            print(f"SegFormer CSV : {seg_source.resolve()}")
            print("-" * 80)
            print(f"{'Metric':<15} {'Method3 (U-Net)':<22} {'SegFormer':<22} {'Diff(S-M3)':<12}")
            print("-" * 80)

            metrics = [
                ('Eyelid', 'Eyelid_mean', 'Eyelid_std'),
                ('Iris', 'Iris_mean', 'Iris_std'),
                ('Pupil', 'Pupil_mean', 'Pupil_std'),
                ('Mean', 'Total_mean', 'Total_std'),
            ]

            for name, mean_key, std_key in metrics:
                m3_mean = float(method3_row[mean_key])
                m3_std = float(method3_row[std_key])
                seg_mean = float(summary[mean_key])
                seg_std = float(summary[std_key])
                diff = seg_mean - m3_mean
                print(f"{name:<15} {m3_mean:.4f} ± {m3_std:.4f}      {seg_mean:.4f} ± {seg_std:.4f}      {diff:+.4f}")

            print("=" * 80)

            comparison = {
                'Method': ['Method3 (U-Net)', f"SegFormer ({SEGFORMER_COMPARE_MODE})"],
                'Eyelid_mean': [float(method3_row['Eyelid_mean']), summary['Eyelid_mean']],
                'Eyelid_std': [float(method3_row['Eyelid_std']), summary['Eyelid_std']],
                'Iris_mean': [float(method3_row['Iris_mean']), summary['Iris_mean']],
                'Iris_std': [float(method3_row['Iris_std']), summary['Iris_std']],
                'Pupil_mean': [float(method3_row['Pupil_mean']), summary['Pupil_mean']],
                'Pupil_std': [float(method3_row['Pupil_std']), summary['Pupil_std']],
                'Total_mean': [float(method3_row['Total_mean']), summary['Total_mean']],
                'Total_std': [float(method3_row['Total_std']), summary['Total_std']],
            }
            comparison_df = pd.DataFrame(comparison)
            out_path = results_dir / f"ablation_comparison_{timestamp}.csv"
            comparison_df.to_csv(out_path, index=False)
            print(f"\n✅ 比較結果を保存しました: {out_path.resolve()}")



📊 Method3 vs SegFormer 比較
Metric          Method3 (U-Net)      SegFormer            差分             
--------------------------------------------------------------------------------


NameError: name 'summary' is not defined

In [ ]:
# Method3（U-Net）vs SegFormer 比較（セル単独実行でも動くように self-contained）
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Method3（U-Net）側：最新の summary CSV ----
method3_files = sorted(results_dir.glob("cv_eval_summary_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not method3_files:
    print("⚠️ Method3の結果ファイル（results/cv_eval_summary_*.csv）が見つかりません。")
else:
    method3_results = pd.read_csv(method3_files[0])
    if 'Method' not in method3_results.columns or 'Method3' not in set(method3_results['Method'].astype(str)):
        print(f"⚠️ Method3行が見つかりません: {method3_files[0]}")
    else:
        method3_row = method3_results[method3_results['Method'].astype(str) == 'Method3'].iloc[0]

        # ---- SegFormer側：どの手法の結果と比べるか ----
        # raw / outerarc / fullmax / ransac_whole / ransac_arc
        SEGFORMER_COMPARE_MODE = 'fullmax'

        # まず per-image CSV（推奨）から平均±SD（画像単位）を作る。無ければfold平均CSVにfallback。
        per_path = sorted(results_dir.glob(f"segformer_eval_perimage_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
        fold_path = sorted(results_dir.glob(f"segformer_eval_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

        seg_source = None
        if per_path:
            df_seg = pd.read_csv(per_path[0])
            seg_source = per_path[0]
        elif fold_path:
            df_seg = pd.read_csv(fold_path[0])
            seg_source = fold_path[0]
        else:
            df_seg = None

        if df_seg is None:
            print(f"⚠️ SegFormerの結果CSVが見つかりません: mode={SEGFORMER_COMPARE_MODE}")
        else:
            # df_seg は列: eyelid/iris/pupil/mean を想定
            for col in ['eyelid','iris','pupil','mean']:
                if col not in df_seg.columns:
                    raise ValueError(f"SegFormer CSVに列 '{col}' がありません: {seg_source}")

            summary = {
                'Eyelid_mean': float(df_seg['eyelid'].mean()),
                'Eyelid_std': float(df_seg['eyelid'].std(ddof=1)),
                'Iris_mean': float(df_seg['iris'].mean()),
                'Iris_std': float(df_seg['iris'].std(ddof=1)),
                'Pupil_mean': float(df_seg['pupil'].mean()),
                'Pupil_std': float(df_seg['pupil'].std(ddof=1)),
                'Total_mean': float(df_seg['mean'].mean()),
                'Total_std': float(df_seg['mean'].std(ddof=1)),
            }

            print("\nMethod3 vs SegFormer 比較")
            print("=" * 80)
            print(f"SegFormer mode: {SEGFORMER_COMPARE_MODE}")
            print(f"SegFormer CSV : {seg_source.resolve()}")
            print("-" * 80)
            print(f"{'Metric':<15} {'Method3 (U-Net)':<22} {'SegFormer':<22} {'Diff(S-M3)':<12}")
            print("-" * 80)

            metrics = [
                ('Eyelid', 'Eyelid_mean', 'Eyelid_std'),
                ('Iris', 'Iris_mean', 'Iris_std'),
                ('Pupil', 'Pupil_mean', 'Pupil_std'),
                ('Mean', 'Total_mean', 'Total_std'),
            ]

            for name, mean_key, std_key in metrics:
                m3_mean = float(method3_row[mean_key])
                m3_std = float(method3_row[std_key])
                seg_mean = float(summary[mean_key])
                seg_std = float(summary[std_key])
                diff = seg_mean - m3_mean
                print(f"{name:<15} {m3_mean:.4f} ± {m3_std:.4f}      {seg_mean:.4f} ± {seg_std:.4f}      {diff:+.4f}")

            print("=" * 80)

            comparison = {
                'Method': ['Method3 (U-Net)', f"SegFormer ({SEGFORMER_COMPARE_MODE})"],
                'Eyelid_mean': [float(method3_row['Eyelid_mean']), summary['Eyelid_mean']],
                'Eyelid_std': [float(method3_row['Eyelid_std']), summary['Eyelid_std']],
                'Iris_mean': [float(method3_row['Iris_mean']), summary['Iris_mean']],
                'Iris_std': [float(method3_row['Iris_std']), summary['Iris_std']],
                'Pupil_mean': [float(method3_row['Pupil_mean']), summary['Pupil_mean']],
                'Pupil_std': [float(method3_row['Pupil_std']), summary['Pupil_std']],
                'Total_mean': [float(method3_row['Total_mean']), summary['Total_mean']],
                'Total_std': [float(method3_row['Total_std']), summary['Total_std']],
            }
            comparison_df = pd.DataFrame(comparison)
            out_path = results_dir / f"ablation_comparison_{timestamp}.csv"
            comparison_df.to_csv(out_path, index=False)
            print(f"\n✅ 比較結果を保存しました: {out_path.resolve()}")



📊 Method3 vs SegFormer 比較
Metric          Method3 (U-Net)      SegFormer            差分             
--------------------------------------------------------------------------------


NameError: name 'summary' is not defined

In [ ]:
# Method3（U-Net）vs SegFormer 比較（セル単独実行でも動くように self-contained）
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime

results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# ---- Method3（U-Net）側：最新の summary CSV ----
method3_files = sorted(results_dir.glob("cv_eval_summary_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
if not method3_files:
    print("⚠️ Method3の結果ファイル（results/cv_eval_summary_*.csv）が見つかりません。")
else:
    method3_results = pd.read_csv(method3_files[0])
    if 'Method' not in method3_results.columns or 'Method3' not in set(method3_results['Method'].astype(str)):
        print(f"⚠️ Method3行が見つかりません: {method3_files[0]}")
    else:
        method3_row = method3_results[method3_results['Method'].astype(str) == 'Method3'].iloc[0]

        # ---- SegFormer側：どの手法の結果と比べるか ----
        # raw / outerarc / fullmax / ransac_whole / ransac_arc
        SEGFORMER_COMPARE_MODE = 'fullmax'

        # まず per-image CSV（推奨）から平均±SD（画像単位）を作る。無ければfold平均CSVにfallback。
        per_path = sorted(results_dir.glob(f"segformer_eval_perimage_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
        fold_path = sorted(results_dir.glob(f"segformer_eval_{SEGFORMER_COMPARE_MODE}_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)

        seg_source = None
        if per_path:
            df_seg = pd.read_csv(per_path[0])
            seg_source = per_path[0]
        elif fold_path:
            df_seg = pd.read_csv(fold_path[0])
            seg_source = fold_path[0]
        else:
            df_seg = None

        if df_seg is None:
            print(f"⚠️ SegFormerの結果CSVが見つかりません: mode={SEGFORMER_COMPARE_MODE}")
        else:
            # df_seg は列: eyelid/iris/pupil/mean を想定
            for col in ['eyelid','iris','pupil','mean']:
                if col not in df_seg.columns:
                    raise ValueError(f"SegFormer CSVに列 '{col}' がありません: {seg_source}")

            summary = {
                'Eyelid_mean': float(df_seg['eyelid'].mean()),
                'Eyelid_std': float(df_seg['eyelid'].std(ddof=1)),
                'Iris_mean': float(df_seg['iris'].mean()),
                'Iris_std': float(df_seg['iris'].std(ddof=1)),
                'Pupil_mean': float(df_seg['pupil'].mean()),
                'Pupil_std': float(df_seg['pupil'].std(ddof=1)),
                'Total_mean': float(df_seg['mean'].mean()),
                'Total_std': float(df_seg['mean'].std(ddof=1)),
            }

            print("\nMethod3 vs SegFormer 比較")
            print("=" * 80)
            print(f"SegFormer mode: {SEGFORMER_COMPARE_MODE}")
            print(f"SegFormer CSV : {seg_source.resolve()}")
            print("-" * 80)
            print(f"{'Metric':<15} {'Method3 (U-Net)':<22} {'SegFormer':<22} {'Diff(S-M3)':<12}")
            print("-" * 80)

            metrics = [
                ('Eyelid', 'Eyelid_mean', 'Eyelid_std'),
                ('Iris', 'Iris_mean', 'Iris_std'),
                ('Pupil', 'Pupil_mean', 'Pupil_std'),
                ('Mean', 'Total_mean', 'Total_std'),
            ]

            for name, mean_key, std_key in metrics:
                m3_mean = float(method3_row[mean_key])
                m3_std = float(method3_row[std_key])
                seg_mean = float(summary[mean_key])
                seg_std = float(summary[std_key])
                diff = seg_mean - m3_mean
                print(f"{name:<15} {m3_mean:.4f} ± {m3_std:.4f}      {seg_mean:.4f} ± {seg_std:.4f}      {diff:+.4f}")

            print("=" * 80)

            comparison = {
                'Method': ['Method3 (U-Net)', f"SegFormer ({SEGFORMER_COMPARE_MODE})"],
                'Eyelid_mean': [float(method3_row['Eyelid_mean']), summary['Eyelid_mean']],
                'Eyelid_std': [float(method3_row['Eyelid_std']), summary['Eyelid_std']],
                'Iris_mean': [float(method3_row['Iris_mean']), summary['Iris_mean']],
                'Iris_std': [float(method3_row['Iris_std']), summary['Iris_std']],
                'Pupil_mean': [float(method3_row['Pupil_mean']), summary['Pupil_mean']],
                'Pupil_std': [float(method3_row['Pupil_std']), summary['Pupil_std']],
                'Total_mean': [float(method3_row['Total_mean']), summary['Total_mean']],
                'Total_std': [float(method3_row['Total_std']), summary['Total_std']],
            }
            comparison_df = pd.DataFrame(comparison)
            out_path = results_dir / f"ablation_comparison_{timestamp}.csv"
            comparison_df.to_csv(out_path, index=False)
            print(f"\n✅ 比較結果を保存しました: {out_path.resolve()}")



📊 Method3 vs SegFormer 比較
Metric          Method3 (U-Net)      SegFormer            差分             
--------------------------------------------------------------------------------


NameError: name 'summary' is not defined